# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarfarukh786/FlyRank-task1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook checks two signals, builds one transparent action rule, writes a ranked queue, and reviews the top ten with explicit failure conditions.

## 1. My rule and its reason codes

### Signal checks before the rule

I will test two inputs the rule depends on, using `trend_direction == "down"` only as an audit outcome, never as a score feature:

1. **Staleness (`days_since_last_update`)** is linked to FlyRank refresh flags. If older pages decline more often, the refresh assumption is **CONFIRMED**; otherwise it is a useful warning against the rule.
2. **Visibility (`impressions_90d`)** is the volume floor behind a quick-win queue. If higher-volume pages decline more often, the quick-win assumption is **CONFIRMED**; if the rates do not move consistently, it is **MIXED**.

The rule is: **prioritize pages updated at least 181 days ago that earned at least 100 impressions in the trailing 90 days. Rank those pages by `impressions_90d`, because a refresh is more actionable when the page has measurable search visibility.** The score is transparent and hand-written; it does not use `trend_pct`, `trend_direction`, IDs, or any future-window field.

The only reason code is `stale_visible_refresh`: the page is both stale and visible enough to justify review. The action label is `refresh_content`. Non-qualifying rows remain in the queue with score 0 and action `monitor` so the output covers every content item.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository root in local VS Code or a Colab checkout.
root = Path.cwd()
for candidate in (root, *root.parents):
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        root = candidate
        break
else:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

raw_path = root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(raw_path)
required_columns = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# This label is used only to audit signals, never to build the baseline score.
df["decline_audit"] = df["trend_direction"].eq("down").astype(int)
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "366+"],
    include_lowest=True,
)
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 100, 1_000, 10_000, np.inf],
    labels=["1-100", "101-1,000", "1,001-10,000", "10,001+"],
    include_lowest=True,
)

def signal_verdict(table: pd.DataFrame, direction: str) -> str:
    rates = table["decline_rate"].dropna().to_numpy()
    if len(rates) < 2:
        return "FALSE"
    changes = np.diff(rates)
    if direction == "up" and np.all(changes >= 0) and changes[-1] > 0:
        return "CONFIRMED"
    if direction == "down" and np.all(changes <= 0) and changes[-1] < 0:
        return "CONFIRMED"
    if np.all(changes <= 0) or np.all(changes >= 0):
        return "OPPOSITE"
    return "MIXED"

staleness_table = (
    df.groupby("staleness_bucket", observed=False)["decline_audit"]
    .agg(n="size", decline_rate="mean")
    .reset_index()
)
volume_table = (
    df.groupby("volume_bucket", observed=False)["decline_audit"]
    .agg(n="size", decline_rate="mean")
    .reset_index()
)

print("Signal 1: staleness bucket table (n and observed decline rate)")
print(staleness_table.to_string(index=False, formatters={"decline_rate": "{:.3f}".format}))
print(f"Verdict: {signal_verdict(staleness_table, 'up')}")
print()
print("Signal 2: visibility/volume bucket table (n and observed decline rate)")
print(volume_table.to_string(index=False, formatters={"decline_rate": "{:.3f}".format}))
print(f"Verdict: {signal_verdict(volume_table, 'up')}")
print()
print(f"Rows: {len(df):,}; observed decline base rate: {df['decline_audit'].mean():.3f}")

Signal 1: staleness bucket table (n and observed decline rate)
staleness_bucket     n decline_rate
            0-90 20655        0.512
          91-180  9171        0.611
         181-365   169        0.467
            366+     5        0.600
Verdict: MIXED

Signal 2: visibility/volume bucket table (n and observed decline rate)
volume_bucket    n decline_rate
        1-100 8006        0.389
    101-1,000 8485        0.603
 1,001-10,000 9907        0.620
      10,001+ 3602        0.524
Verdict: MIXED

Rows: 30,000; observed decline base rate: 0.542


## 2. Build the ranked queue (writes the CSV)

The score is deliberately not a fitted model: `stale_flag * visible_flag * log1p(impressions_90d)`. The binary flags make the eligibility rule readable, while the log keeps very large impression counts from completely dominating the ranking. The CSV is regenerated from this notebook at `work/outputs/baseline_action_score.csv`.

In [6]:
stale_flag = df["days_since_last_update"].ge(181)
visible_flag = df["impressions_90d"].ge(100)

# The score uses only snapshot fields available before any future outcome.
df["score"] = (
    stale_flag.astype(int)
    * visible_flag.astype(int)
    * np.log1p(df["impressions_90d"])
)
df["reason_code"] = np.where(
    (stale_flag & visible_flag),
    "stale_visible_refresh",
    "no_refresh_flag",
)
df["action_label"] = np.where(
    (stale_flag & visible_flag),
    "refresh_content",
    "monitor",
)

def precision_at_k(frame: pd.DataFrame, k: int) -> float:
    return float(frame.sort_values(["score", "content_id"], ascending=[False, True]).head(k)["decline_audit"].mean())

ranked = df.sort_values(["score", "content_id"], ascending=[False, True]).reset_index(drop=True)
ranked["rank"] = np.arange(1, len(ranked) + 1)
output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
]
output_path = root / "work" / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
ranked[output_columns].to_csv(output_path, index=False)

print(f"Scored {len(ranked):,} rows; flagged {int((ranked['action_label'] == 'refresh_content').sum()):,} for refresh")
print(f"Wrote {output_path.relative_to(root)}")
print(f"Observed decline base rate: {df['decline_audit'].mean():.3f}")
print(f"Observed precision@10 (audit only, not a feature): {precision_at_k(ranked, 10):.3f}")
print("Reason codes:", ranked["reason_code"].value_counts().to_dict())

Scored 30,000 rows; flagged 35 for refresh
Wrote work\outputs\baseline_action_score.csv
Observed decline base rate: 0.542
Observed precision@10 (audit only, not a feature): 1.000
Reason codes: {'no_refresh_flag': 29965, 'stale_visible_refresh': 35}


## 3. Top-10 review

The following cell prints one review line per ranked item. The confidence note is deliberately modest: the signal audit was MIXED, so this is a prioritization queue, not a claim that every page needs a refresh.

In [7]:
top10 = ranked.head(10).copy()
print("Top-10 skeptic review")
for _, row in top10.iterrows():
    print(
        f"{int(row['rank'])}. {row['content_id']} | action={row['action_label']} | "
        f"reason={row['reason_code']} | confidence=moderate, because it is stale and visible | "
        f"wrong if the update date is inaccurate, impressions are low-quality, or the page already recovered."
    )

assert len(top10) == 10
assert top10["score"].is_monotonic_decreasing
assert set(ranked["reason_code"]) == {"stale_visible_refresh", "no_refresh_flag"}

Top-10 skeptic review
1. content_cf56e2e2e282 | action=refresh_content | reason=stale_visible_refresh | confidence=moderate, because it is stale and visible | wrong if the update date is inaccurate, impressions are low-quality, or the page already recovered.
2. content_7368877ea310 | action=refresh_content | reason=stale_visible_refresh | confidence=moderate, because it is stale and visible | wrong if the update date is inaccurate, impressions are low-quality, or the page already recovered.
3. content_1bfaa38ff26c | action=refresh_content | reason=stale_visible_refresh | confidence=moderate, because it is stale and visible | wrong if the update date is inaccurate, impressions are low-quality, or the page already recovered.
4. content_0a91db491d14 | action=refresh_content | reason=stale_visible_refresh | confidence=moderate, because it is stale and visible | wrong if the update date is inaccurate, impressions are low-quality, or the page already recovered.
5. content_5feee3994adb | acti

## 4. Weak picks + leakage check

The audit is intentionally skeptical: the rule can surface a page because it is old and visible, even when the volume is modest or the decline relationship is not stable. Those are review candidates, not automatic edits. The final assertions document that the score did not use the label source or a future window.

In [8]:
weak_picks = top10.nsmallest(3, "impressions_90d")
print("Weak picks to verify manually")
for _, row in weak_picks.iterrows():
    print(
        f"rank {int(row['rank'])}: {row['content_id']} has {int(row['impressions_90d']):,} impressions; "
        "it may be too small an opportunity despite passing the visibility floor."
    )

score_inputs = {"days_since_last_update", "impressions_90d"}
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "decline_audit",
    "impressions_last_30d",
    "impressions_prev_30d",
}
assert score_inputs.isdisjoint(forbidden_inputs)
assert output_path.exists()
assert len(pd.read_csv(output_path)) == len(df)
print("Leakage check: PASS; score inputs are snapshot staleness and volume only.")

Weak picks to verify manually
rank 10: content_928af3e22c80 has 1,697 impressions; it may be too small an opportunity despite passing the visibility floor.
rank 9: content_ecb6215e79fd has 4,429 impressions; it may be too small an opportunity despite passing the visibility floor.
rank 8: content_fe16a55cd13d has 4,556 impressions; it may be too small an opportunity despite passing the visibility floor.
Leakage check: PASS; score inputs are snapshot staleness and volume only.


## Self-check

- [x] Two signal checks have visible bucket tables, `n`, and one-word verdicts.
- [x] The rule has one score, one reason code, and an action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] Ten ranked rows have an action, rationale, confidence note, and what would make each wrong.
- [x] The score excludes labels, IDs, `trend_pct`, `trend_direction`, and future comparison windows.
- [ ] Commit this notebook and submit the repository URL.